## Demo Overview

This notebook demonstrates Mongolian ASR inference using a NeMo model fine-tuned on Mongolian speech data.

- **Base model:** `nvidia/stt_en_fastconformer_hybrid_large_streaming_multi`
- **Fine-tuned model used here:** `models/final_mn_finetuned.nemo`
- **Tokenizer:** SentencePiece (`unigram`), vocab size `1024`
- **Dataset:** [Mongolian Speech Dataset (Kaggle)](https://www.kaggle.com/datasets/otgonbaatar11/mongolian-speech-dataset)
- **Training epochs:** `3` (CPU-only training, couldn't afford GPU)
- **Audio sample rate:** `16,000 Hz`
- **Inference batch size:** `1`
- **Input modes:** `wav`, `mic`, `mic_live`
- **Decoder:** `auto` (on CPU, it may fall back to `ctc` for better stability)

### Notes

- This is a low-resource demo, so transcription quality can be limited.
- Fine-tuning logic is in `mn_finetune_setup.py` and `mn_finetune_train.py`.


## 1) Imports and Runtime Checks

In [1]:
from __future__ import annotations

import math
import os
import sys
import tempfile
import time
from pathlib import Path
from typing import Optional, Tuple
import contextlib
import io
import warnings
import queue

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
import torch

import nemo.collections.asr as nemo_asr
from nemo.utils import logging as nemo_logging

try:
    import sounddevice as sd
    HAS_SOUNDDEVICE = True
except Exception as exc:
    HAS_SOUNDDEVICE = False
    SD_IMPORT_ERROR = str(exc)

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not HAS_SOUNDDEVICE:
    print("sounddevice not available:", SD_IMPORT_ERROR)


[NeMo W 2026-03-11 19:22:47 <frozen importlib:241] Megatron num_microbatches_calculator not found, using Apex version.
W0311 19:22:47.904000 27604 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-03-11 19:22:53 warnings:109] c:\Users\codet\Documents\Playground\.speech_venv\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
      warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
    


PyTorch: 2.10.0+cpu
CUDA available: False


## 2) Configuration

- `model_ref` can be:
  - local `.nemo` path
  - NeMo pretrained name
  - Azure ML data asset reference: `azureml:<asset_name>:<version>`
- `stream_mode="accumulate"` usually gives smoother partial text.


In [2]:
CONFIG = {
    "model_ref": "models/final_mn_finetuned.nemo",
    "input_mode": "mic_live",  # "wav" or "mic"
    "wav_path": "./audios/MB145-0001.wav",
    "mic_seconds": 20,
    "mic_device_index": None,
    "sample_rate": 16000,
    "batch_size": 1,
    "decoder": "auto",  # auto | rnnt | ctc
    "chunk_ms": 250,
    "decode_every_n_chunks": 2,
    "stream_mode": "accumulate",      # accumulate | chunk_only
    "stream_render_mode": "chatgpt",  # chatgpt | verbose
    "simulate_realtime": False,
    "show_partial_each_chunk": False,
    "nemo_log_level": "ERROR",
    "azureml_env_file": ".env",
    "azureml_cache_dir": ".azureml_model_cache",
    "output_text_file": "./mn_finetune_assets/infer_outputs_notebook.txt",
}

CONFIG

{'model_ref': 'models/final_mn_finetuned.nemo',
 'input_mode': 'mic_live',
 'wav_path': './audios/MB145-0001.wav',
 'mic_seconds': 20,
 'mic_device_index': None,
 'sample_rate': 16000,
 'batch_size': 1,
 'decoder': 'auto',
 'chunk_ms': 250,
 'decode_every_n_chunks': 2,
 'stream_mode': 'accumulate',
 'stream_render_mode': 'chatgpt',
 'simulate_realtime': False,
 'show_partial_each_chunk': False,
 'nemo_log_level': 'ERROR',
 'azureml_env_file': '.env',
 'azureml_cache_dir': '.azureml_model_cache',
 'output_text_file': './mn_finetune_assets/infer_outputs_notebook.txt'}

## 3) Utilities (dotenv + Azure asset resolution + audio helpers)

In [ ]:
def resolve_model_ref(model_ref: str) -> str:
    if model_ref:
        return model_ref
    return str(Path("models/final_mn_finetuned.nemo").resolve())


def normalize_audio_mono_float32(audio: np.ndarray) -> np.ndarray:
    audio = np.asarray(audio)
    if audio.ndim == 2:
        audio = np.mean(audio, axis=1)
    if np.issubdtype(audio.dtype, np.integer):
        audio = audio.astype(np.float32) / float(np.iinfo(audio.dtype).max)
    else:
        audio = audio.astype(np.float32)
    return np.clip(audio, -1.0, 1.0)


def resample_audio(audio: np.ndarray, src_sr: int, dst_sr: int) -> np.ndarray:
    if src_sr == dst_sr:
        return audio.astype(np.float32)
    g = math.gcd(src_sr, dst_sr)
    return resample_poly(audio, dst_sr // g, src_sr // g).astype(np.float32)


def load_wav_for_model(path: Path, target_sr: int) -> np.ndarray:
    if not path.exists():
        raise FileNotFoundError(path)
    audio, sr = sf.read(str(path), always_2d=False)
    audio = normalize_audio_mono_float32(audio)
    return resample_audio(audio, sr, target_sr)


def record_microphone(seconds: float, sample_rate: int, device: Optional[int]) -> np.ndarray:
    if not HAS_SOUNDDEVICE:
        raise RuntimeError("sounddevice is not installed. Install with: python -m pip install sounddevice")
    n = int(max(1, round(seconds * sample_rate)))
    print(f"Recording {seconds:.1f}s at {sample_rate} Hz...")
    audio = sd.rec(n, samplerate=sample_rate, channels=1, dtype="float32", device=device)
    sd.wait()
    return np.asarray(audio[:, 0], dtype=np.float32)


In [4]:
def stream_from_microphone_live(
    model,
    seconds: float,
    sample_rate: int,
    chunk_samples: int,
    batch_size: int = 1,
    decoder_mode: str = "auto",
    mic_device_index: Optional[int] = None,
    decode_every_n_chunks: int = 2,
):
    if not HAS_SOUNDDEVICE:
        raise RuntimeError("sounddevice is not installed")

    from IPython.display import Markdown, display

    # On CPU, auto->RNNT can be unstable for live partials. Force CTC unless user overrides.
    effective_decoder = decoder_mode
    if decoder_mode == "auto" and not torch.cuda.is_available():
        effective_decoder = "ctc"

    def _lcp_len(a: str, b: str) -> int:
        n = min(len(a), len(b))
        i = 0
        while i < n and a[i] == b[i]:
            i += 1
        return i

    q = queue.Queue()
    chunks = []
    used_decoder = "unknown"
    prev_partial = ""
    committed = ""
    best_text = ""
    seen = 0

    def _cb(indata, frames, time_info, status):
        q.put(np.asarray(indata[:, 0], dtype=np.float32).copy())

    handle = display(Markdown("**Assistant (live)**\n\n..."), display_id=True)
    start = time.time()

    with sd.InputStream(
        samplerate=sample_rate,
        channels=1,
        dtype="float32",
        blocksize=chunk_samples,
        device=mic_device_index,
        callback=_cb,
    ):
        while (time.time() - start) < seconds:
            try:
                chunk = q.get(timeout=0.2)
            except queue.Empty:
                continue

            chunks.append(chunk)
            seen += 1
            if seen % max(1, int(decode_every_n_chunks)) != 0:
                continue

            running_audio = np.concatenate(chunks, axis=0)
            partial, used_decoder = transcribe_with_decoder(
                model,
                running_audio,
                sample_rate,
                batch_size=batch_size,
                decoder_mode=effective_decoder,
            )
            partial = str(partial).strip()

            # Commit only text that stays stable across consecutive partials.
            lcp = _lcp_len(prev_partial, partial)
            if prev_partial.startswith(committed) and lcp > len(committed):
                committed += prev_partial[len(committed):lcp]

            if partial.startswith(committed):
                live_text = (committed + partial[len(committed):]).strip()
            else:
                live_text = (committed + " " + partial).strip() if committed else partial

            if len(live_text) > len(best_text):
                best_text = live_text

            elapsed = min(time.time() - start, seconds)
            handle.update(
                Markdown(
                    f"**Assistant (live)** `{elapsed:.2f}s / {seconds:.2f}s` (decoder={used_decoder})\n\n"
                    f"{live_text if live_text else '...'}"
                )
            )
            prev_partial = partial

    # Final flush: prefer the longest coherent hypothesis we saw.
    final_text = best_text.strip()
    if prev_partial:
        if prev_partial.startswith(committed):
            candidate = (committed + prev_partial[len(committed):]).strip()
        else:
            candidate = (committed + " " + prev_partial).strip() if committed else prev_partial
        if len(candidate) >= len(final_text):
            final_text = candidate

    audio = np.concatenate(chunks, axis=0) if chunks else np.zeros(0, dtype=np.float32)
    return final_text, used_decoder, audio


## 4) Model Loading + Decoding Helpers

In [5]:
def transcribe_audio_array(model, audio: np.ndarray, sr: int, batch_size: int = 1) -> str:
    tmp_wav = None
    try:
        with tempfile.NamedTemporaryFile(suffix="_asr.wav", delete=False) as f:
            tmp_wav = f.name
        sf.write(tmp_wav, audio, sr)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                out = model.transcribe([tmp_wav], batch_size=batch_size)
        if isinstance(out, (list, tuple)) and len(out) > 0:
            item = out[0]
            return item.text if hasattr(item, "text") else str(item)
        return ""
    finally:
        if tmp_wav and os.path.exists(tmp_wav):
            try:
                os.remove(tmp_wav)
            except Exception:
                pass


def _set_decoder_type(model, decoder_type: str) -> bool:
    if not hasattr(model, "change_decoding_strategy"):
        return False
    try:
        model.change_decoding_strategy(decoder_type=decoder_type, verbose=False)
        return True
    except TypeError:
        try:
            model.change_decoding_strategy(decoder_type=decoder_type)
            return True
        except Exception:
            return False
    except Exception:
        return False


def transcribe_with_decoder(model, audio: np.ndarray, sr: int, batch_size: int, decoder_mode: str):
    def _run() -> str:
        return transcribe_audio_array(model, audio, sr, batch_size=batch_size)

    has_hybrid_decoders = hasattr(model, "ctc_decoder") and hasattr(model, "joint")
    current = str(getattr(model, "cur_decoder", "unknown"))

    if decoder_mode == "rnnt":
        _set_decoder_type(model, "rnnt")
        return _run(), "rnnt"
    if decoder_mode == "ctc":
        _set_decoder_type(model, "ctc")
        return _run(), "ctc"

    if has_hybrid_decoders:
        _set_decoder_type(model, "rnnt")
        rnnt_text = _run()
        if str(rnnt_text).strip():
            return rnnt_text, "rnnt"
        if _set_decoder_type(model, "ctc"):
            ctc_text = _run()
            return ctc_text, "ctc"
        return rnnt_text, "rnnt"

    return _run(), current


def load_model(model_ref: str):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    p = Path(model_ref)
    if p.exists() and p.suffix.lower() == ".nemo":
        print(f"Loading local .nemo model: {p.resolve()}")
        model = nemo_asr.models.ASRModel.restore_from(str(p.resolve()), map_location=device)
    else:
        print(f"Loading pretrained model by name: {model_ref}")
        model = nemo_asr.models.ASRModel.from_pretrained(model_name=model_ref)
    model = model.to(device)
    model.eval()
    print(f"Device: {device}")
    return model


## 5) Chunk Construction (inspect how audio is split)

In [6]:
def iter_chunks(audio: np.ndarray, chunk_samples: int):
    for start in range(0, len(audio), chunk_samples):
        end = min(start + chunk_samples, len(audio))
        yield start, end, audio[start:end]


def preview_chunks(audio: np.ndarray, sr: int, chunk_samples: int, max_rows: int = 8):
    total = len(audio)
    rows = []
    for i, (start, end, chunk) in enumerate(iter_chunks(audio, chunk_samples), start=1):
        rows.append((i, start / sr, end / sr, len(chunk) / sr, len(chunk)))
    print(f"Total samples: {total} ({total / sr:.2f}s)")
    print(f"Chunk size: {chunk_samples} samples ({chunk_samples / sr:.3f}s)")
    print(f"Chunk count: {len(rows)}")
    print("\nFirst chunks:")
    for r in rows[:max_rows]:
        idx, s, e, d, n = r
        print(f"  chunk {idx:03d}: {s:7.3f}s -> {e:7.3f}s | dur={d:6.3f}s | samples={n}")
    if len(rows) > max_rows:
        print(f"  ... ({len(rows) - max_rows} more)")


## 6) Streaming-Style Transcription

This prints partial transcript updates chunk by chunk.

- `accumulate`: transcribe all audio received so far (more stable partials)
- `chunk_only`: transcribe current chunk only (fast but less coherent)


In [7]:
def longest_common_prefix_len(a: str, b: str) -> int:
    n = min(len(a), len(b))
    i = 0
    while i < n and a[i] == b[i]:
        i += 1
    return i


def stream_transcribe_chunks(
    model,
    audio: np.ndarray,
    sr: int,
    chunk_samples: int,
    batch_size: int = 1,
    decoder_mode: str = "auto",
    stream_mode: str = "accumulate",
    simulate_realtime: bool = False,
    show_partial_each_chunk: bool = False,
    render_mode: str = "chatgpt",  # chatgpt | verbose
):
    if stream_mode not in {"accumulate", "chunk_only"}:
        raise ValueError("stream_mode must be 'accumulate' or 'chunk_only'")
    if render_mode not in {"chatgpt", "verbose"}:
        raise ValueError("render_mode must be 'chatgpt' or 'verbose'")

    total_chunks = int(math.ceil(len(audio) / max(1, chunk_samples)))
    total_sec = len(audio) / sr

    running = np.zeros(0, dtype=np.float32)
    previous_partial = ""
    used_decoder = "unknown"

    display_handle = None
    Markdown = None

    if render_mode == "chatgpt":
        try:
            from IPython.display import Markdown as _Markdown, display
            Markdown = _Markdown
            display_handle = display(_Markdown("**Assistant (streaming)**\n\n..."), display_id=True)
        except Exception:
            display_handle = None

    if render_mode == "verbose":
        print("Streaming transcript:")
        print("-" * 60)

    for i, (_, _, chunk) in enumerate(iter_chunks(audio, chunk_samples), start=1):
        current = chunk if stream_mode == "chunk_only" else np.concatenate([running, chunk], axis=0)
        if stream_mode == "accumulate":
            running = current

        partial, used_decoder = transcribe_with_decoder(
            model,
            current,
            sr,
            batch_size=batch_size,
            decoder_mode=decoder_mode,
        )
        partial = str(partial).strip()
        done_sec = min(i * chunk_samples / sr, total_sec)

        if render_mode == "verbose":
            if show_partial_each_chunk:
                print(f"[chunk {i}/{total_chunks}] {done_sec:.2f}s/{total_sec:.2f}s decoder={used_decoder}")
                print(f"partial: {partial}")
            else:
                lcp = longest_common_prefix_len(previous_partial, partial)
                delta = partial[lcp:]
                if delta:
                    print(delta, end="", flush=True)
        else:
            msg = (
                f"**Assistant (streaming)**  `{done_sec:.2f}s / {total_sec:.2f}s`\n\n"
                f"{partial if partial else '...'}"
            )
            if display_handle is not None and Markdown is not None:
                display_handle.update(Markdown(msg))
            else:
                print("\r" + (partial if partial else "...") + " " * 8, end="", flush=True)

        previous_partial = partial

        if simulate_realtime:
            time.sleep(len(chunk) / sr)

    if render_mode == "verbose":
        print("\n" + "-" * 60)
    elif display_handle is None:
        print()

    return previous_partial, used_decoder


## 7) Run End-to-End

In [ ]:
if hasattr(nemo_logging, "set_verbosity"):
    nemo_logging.set_verbosity(getattr(nemo_logging, CONFIG["nemo_log_level"]))

model_ref = resolve_model_ref(CONFIG["model_ref"])
model = load_model(model_ref)

chunk_samples = int(CONFIG["sample_rate"] * CONFIG["chunk_ms"] / 1000)
if chunk_samples <= 0:
    raise ValueError("chunk_ms too small; chunk_samples <= 0")

if CONFIG["input_mode"] == "wav":
    audio = load_wav_for_model(Path(CONFIG["wav_path"]), CONFIG["sample_rate"])
    preview_chunks(audio, CONFIG["sample_rate"], chunk_samples)

    final_text, used_decoder = stream_transcribe_chunks(
        model=model,
        audio=audio,
        sr=int(CONFIG["sample_rate"]),
        chunk_samples=chunk_samples,
        batch_size=int(CONFIG["batch_size"]),
        decoder_mode=str(CONFIG["decoder"]),
        stream_mode=str(CONFIG["stream_mode"]),
        simulate_realtime=bool(CONFIG["simulate_realtime"]),
        show_partial_each_chunk=bool(CONFIG["show_partial_each_chunk"]),
        render_mode=str(CONFIG["stream_render_mode"]),
    )

elif CONFIG["input_mode"] == "mic_live":
    # Requires stream_from_microphone_live(...) cell to be defined.
    final_text, used_decoder, audio = stream_from_microphone_live(
        model=model,
        seconds=float(CONFIG["mic_seconds"]),
        sample_rate=int(CONFIG["sample_rate"]),
        chunk_samples=chunk_samples,
        batch_size=int(CONFIG["batch_size"]),
        decoder_mode=str(CONFIG["decoder"]),
        mic_device_index=CONFIG["mic_device_index"],
        decode_every_n_chunks=int(CONFIG["decode_every_n_chunks"]),
    )

elif CONFIG["input_mode"] == "mic":
    # Fallback: records first, then streams from buffered audio.
    audio = record_microphone(
        seconds=float(CONFIG["mic_seconds"]),
        sample_rate=int(CONFIG["sample_rate"]),
        device=CONFIG["mic_device_index"],
    )
    preview_chunks(audio, CONFIG["sample_rate"], chunk_samples)

    final_text, used_decoder = stream_transcribe_chunks(
        model=model,
        audio=audio,
        sr=int(CONFIG["sample_rate"]),
        chunk_samples=chunk_samples,
        batch_size=int(CONFIG["batch_size"]),
        decoder_mode=str(CONFIG["decoder"]),
        stream_mode=str(CONFIG["stream_mode"]),
        simulate_realtime=bool(CONFIG["simulate_realtime"]),
        show_partial_each_chunk=bool(CONFIG["show_partial_each_chunk"]),
        render_mode=str(CONFIG["stream_render_mode"]),
    )

else:
    raise ValueError("CONFIG['input_mode'] must be one of: 'wav', 'mic_live', 'mic'")

print("\nFinal transcript:")
print(final_text)
print("Decoder used:", used_decoder)

out_path = Path(CONFIG["output_text_file"])
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(final_text + "\n", encoding="utf-8")
print("Saved final transcript to:", out_path.resolve())


Using cached Azure model file: C:\Users\codet\Documents\Playground\.azureml_model_cache\fine-tuned-nemo-model\1\final_mn_finetuned.nemo
Loading local .nemo model: C:\Users\codet\Documents\Playground\.azureml_model_cache\fine-tuned-nemo-model\1\final_mn_finetuned.nemo
Device: cpu


**Assistant (live)** `20.00s / 20.00s` (decoder=ctc)

Х намайг би хар тааас.

[NeMo W 2026-03-11 19:24:58 dataloader:335] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-03-11 19:24:58 dataloader:341] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
[NeMo W 2026-03-11 19:25:00 dataloader:335] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-03-11 19:25:00 dataloader:341] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is


Final transcript:
Х намайг би хар тааас ёстой.
Decoder used: ctc
Saved final transcript to: C:\Users\codet\Documents\Playground\mn_finetune_assets\infer_outputs_notebook.txt


## 8) Optional: Quick Device Listing for Mic Mode

In [9]:
if HAS_SOUNDDEVICE:
    print("Input devices:")
    for i, dev in enumerate(sd.query_devices()):
        if int(dev.get("max_input_channels", 0)) > 0:
            print(f"  [{i}] {dev['name']} (in_ch={dev['max_input_channels']}, sr={dev['default_samplerate']})")
else:
    print("sounddevice unavailable; mic mode disabled.")


Input devices:
  [0] Microsoft Sound Mapper - Input (in_ch=2, sr=44100.0)
  [1] Microphone Array (Intel® Smart  (in_ch=4, sr=44100.0)
  [2] Headset (EDIFIER W830NB) (in_ch=1, sr=44100.0)
  [6] Primary Sound Capture Driver (in_ch=2, sr=44100.0)
  [7] Microphone Array (Intel® Smart Sound Technology for Digital Microphones) (in_ch=4, sr=44100.0)
  [8] Headset (EDIFIER W830NB) (in_ch=1, sr=44100.0)
  [14] Headset (EDIFIER W830NB) (in_ch=1, sr=16000.0)
  [15] Microphone Array (Intel® Smart Sound Technology for Digital Microphones) (in_ch=2, sr=48000.0)
  [16] Microphone Array 1 (Intel® Smart Sound Technology DMIC Microphone) (in_ch=2, sr=48000.0)
  [17] Microphone Array 2 (Intel® Smart Sound Technology DMIC Microphone) (in_ch=2, sr=48000.0)
  [18] Microphone Array 3 (Intel® Smart Sound Technology DMIC Microphone) (in_ch=4, sr=16000.0)
  [21] PC Speaker (Realtek HD Audio output with SST) (in_ch=2, sr=48000.0)
  [22] Microphone (Realtek HD Audio Mic input) (in_ch=2, sr=44100.0)
  [23] Stereo 